# Interfacing with DDR4 and LPDDR memory on the VCK190

This board hosts a Double Data Rate 4 (**DDR4**) memory controller and 2x Low-Power DDR4 (**LPDDR4**) memory controllers. PYNQ and the Linux operating system live in the DDR4. The two LPDDR4 chips (**LPDDR0** and **LPDDR1**) are not used by Linux and can be adopted in the user design for data storage. In the golden reference design shipped with PYNQ, we expose 4GB of LPDDR0 for access in the processing system. The LPDDR1 memory controller is exposed only for PL read / writes.  

In [ ]:
from pynq import base.BaseOverlay, allocate
import numpy as np

ol = BaseOverlay("base.pdi")

## The Memory Controllers

All memory, including the LPDDR and DDR memory controllers used in the golden reference and base overlay designs, can be accessed from the overlay's memory dictionary.

In [ ]:
ol.mem_dict

Note that the DDR is split across `noc_ddr/C0_DDR_LOW0` and `noc_ddr/C0_DDR_LOW1`. These memories are interfaced with as attributes of `Overlay()`, dropping the `/` from the name: `ol.noc_lpddr0`, `ol.noc_ddrC0_DDR_LOW0` and `ol.noc_ddrC0_DDR_LOW1`. The base address and size of each memory can be accessed using:

In [ ]:
for name, memory in ol.mem_dict.items():
    if "ddr" in name.lower():
        print(f"Memory: {name.replace("/", "")}\n"
              f"\t Base address: {hex(memory['base_address'])}\n"
              f"\t Size: {memory['size'] / 1024**3:.2f} GB")

We can allocate a PYNQ memory buffer in a given target memory controller using `allocate()`'s `target` keyword argument. Below we create a buffer targeting the DDR4 and LPDDR0 memory space.

In [ ]:
dma = ol.axi_dma_0

N = 2048   # DMA Buffer size -> uint32 words (4-byte) x 2048 = 8192-bytes

ddr = allocate((N,), dtype=np.uint32, target=ol.noc_ddrC0_DDR_LOW0)
lpddr0 = allocate((N,), dtype=np.uint32, target=ol.noc_lpddr0)

print(f"DDR buffer phys addr: {ddr.device_address:#x}")
print(f"LPDDR0 buffer phys addr: {lpddr0.device_address:#x}")

The memory buffer addresses should fall in their `target`'s base address.

## Moving Data Between Memory Controllers

In the cells below we demonstrate moving data from DDR &rarr; LPDDR0 and LPDDR0 &rarr; DDR using the DMA.

### DDR4 &rarr; LPDDR0

Below, we fill `ddr` and `lpddr0` with random numbers and zeros, respectively. The DMA is then used to transfer the samples in the DDR buffer to the LPDDR0 memory via the NoC. Following the transfer the memory buffers are compared to see if it was successful.

In [ ]:
ddr[:] = np.random.randint(0, 2**32, N, dtype=np.uint32)
lpddr0[:] = 0

dma.recvchannel.transfer(lpddr0)
dma.sendchannel.transfer(ddr)
dma.sendchannel.wait()
dma.recvchannel.wait()

if np.array_equal(ddr, lpddr0):
    print(f"DMA loopback PASSED ({N} x uint32 = {N*4} bytes)")
else:
    mismatches = np.where(ddr != lpddr0)[0]
    print(f"DMA loopback FAILED — {len(mismatches)}/{N} words differ")
    print(f"  First 8 DDR sent: {ddr[:8]}")
    print(f"  First 8 LPDDR0 received: {lpddr0[:8]}")
    print(f"  First mismatch at index {mismatches[0]}: "
          f"sent={ddr[mismatches[0]]:#x} recv={lpddr0[mismatches[0]]:#x}")

### LPDDR0 &rarr; DDR4

In this test, we instead load `lpddr0` with random numbers, and `ddr` with zeros, to demonstrate movement of data in the other direction (LPDDR0 &rarr; DDR4). Again, the DMA transfer is carried out and the buffers are compared to validate.

In [ ]:
lpddr0[:] = np.random.randint(0, 2**32, N, dtype=np.uint32)
ddr[:] = 0

dma.recvchannel.transfer(ddr)
dma.sendchannel.transfer(lpddr0)
dma.sendchannel.wait()
dma.recvchannel.wait()

if np.array_equal(lpddr0, ddr):
    print(f"DMA loopback PASSED ({N} x uint32 = {N*4} bytes)")
else:
    mismatches = np.where(lpddr0 != ddr)[0]
    print(f"DMA loopback FAILED — {len(mismatches)}/{N} words differ")
    print(f"  First 8 LPDDR0 sent: {lpddr0[:8]}")
    print(f"  First 8 DDR received: {ddr[:8]}")
    print(f"  First mismatch at index {mismatches[0]}: "
          f"sent={lpddr0[mismatches[0]]:#x} recv={ddr[mismatches[0]]:#x}")

## Free Memory Buffers

Memory buffers are freed after use to return the contiguous memory they occupy for reuse.

In [ ]:
ddr.freebuffer()
lpddr0.freebuffer()